In [38]:
import operator
from typing import Annotated, TypedDict

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field


In [39]:
load_dotenv()

True

In [40]:
llm = ChatGoogleGenerativeAI(
    model='gemini-3.5-flash-lite'
)

In [41]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback for the essay")
    score: int = Field(description="Score out of 10", ge=0, le=10)

In [42]:
structured_llm = llm.with_structured_output(EvaluationSchema)

In [43]:
essay = """
Agentic AI
Today AI is everywhere and most people use things like ChatGPT for school or work. Normally when you use AI you just ask it a question and it writes back an answer, then it stops and waits for you to say something else. Now there is a new thing called agentic AI which is basically AI that can do stuff by itself without you having to tell it every single step to take.
The big difference is that regular AI just generates text while agentic AI can actually take action and make its own plans. Like if you ask normal AI to plan a trip it just gives you a list of places, which is okay I guess, but an agentic AI can actually go online, look at flight prices, check your calendar and book things for you. And if something goes wrong like a flight is full it doesn't just stop, it tries to fix the problem on its own and finds another flight.
Lots of businesses are already using this kind of AI because it saves them time. In coding, agentic AI can write code and then test it to see if there are bugs, and if there is an error it fixes it without a human needing to help. It is also used in customer service. Instead of those basic chatbots that don't really do anything, an AI agent can check your order, give you your money back and send an email. This means people don't have to do the boring repetitive jobs as much.
Even though this sounds really cool there are still a lot of problems with it. Since the AI is doing things on its own, if it messes up it could cause a really big issue before anyone even notices. For example if it doesn't understand what you meant it might buy the wrong thing or delete important files by accident. Also giving an AI access to your personal info like your bank account or emails is pretty scary because of hackers and privacy issues.
Overall, agentic AI is a huge step in technology because it turns AI from just a chatbot into something that can actually get things done by itself. There are still some big risks and mistakes that need to be fixed first, so humans still need to keep an eye on it, but in the future it will probably be used by everyone.
"""

In [44]:
prompt = f"Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {essay}"
structured_llm.invoke(prompt)

EvaluationSchema(feedback="The essay is written in a very casual, conversational tone. While the vocabulary is accessible and the explanations of agentic AI are clear, it lacks academic or formal phrasing (e.g., using 'stuff', 'like', 'really big issue'). The grammar and punctuation are generally fine, but sentence structures are repetitive, often relying on run-on sentences connected by 'and' or 'so'. To improve, the writer should use more formal transitions, richer vocabulary, and vary sentence structures for better flow.", score=6)

In [45]:
class UPSCState(TypedDict):

    essay: str
    language_feedback: str
    analysis_feedback: str
    clarity_feedback: str
    overall_feedback: str

    individual_scores: Annotated[list[int], operator.add]
    avg_score: float


In [46]:
def evaluate_language(state: UPSCState):

    prompt = f"Evaluate the language quality of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}"
    output = structured_llm.invoke(prompt)
    return {'language_feedback': output.feedback, 'individual_scores': [output.score]}

In [47]:
def evaluate_analysis(state: UPSCState):

    prompt = f"Evaluate the depth of analysis of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}"
    output = structured_llm.invoke(prompt)
    return {'analysis_feedback': output.feedback, 'individual_scores': [output.score]}

In [48]:
def evaluate_thought(state: UPSCState):

    prompt = f"Evaluate the clarity of thought of the following essay and provide a feedback and assign a score out of 10 \n {state['essay']}"
    output = structured_llm.invoke(prompt)
    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

In [49]:
def final_evaluation(state: UPSCState):
    # summary feedback
    prompt = f"Based on the following feedbacks create a summarized feedback \n language feedback - {state['language_feedback']} \n depth of analysis feedback - {state['analysis_feedback']} \n clarity of thought feedback - {state['clarity_feedback']}"
    chain = llm | StrOutputParser()
    overall_feedback = chain.invoke(prompt)

    # average calculate
    avg_score = sum(state['individual_scores'])/len(state['individual_scores'])
    return {'overall_feedback': overall_feedback, 'avg_score': avg_score}

In [50]:
graph = StateGraph(UPSCState)

# Nodes
graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

# Edges
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)

# Compile
workflow = graph.compile()

In [51]:
initial_state = {
    'essay': essay
}
workflow.invoke(initial_state)

{'essay': "\nAgentic AI\nToday AI is everywhere and most people use things like ChatGPT for school or work. Normally when you use AI you just ask it a question and it writes back an answer, then it stops and waits for you to say something else. Now there is a new thing called agentic AI which is basically AI that can do stuff by itself without you having to tell it every single step to take.\nThe big difference is that regular AI just generates text while agentic AI can actually take action and make its own plans. Like if you ask normal AI to plan a trip it just gives you a list of places, which is okay I guess, but an agentic AI can actually go online, look at flight prices, check your calendar and book things for you. And if something goes wrong like a flight is full it doesn't just stop, it tries to fix the problem on its own and finds another flight.\nLots of businesses are already using this kind of AI because it saves them time. In coding, agentic AI can write code and then test 